# Joint N/V LightGCN 0단계 체크포인트 진단

기존 H&M 60일 seed 42 `joint_nv` 체크포인트를 재학습하지 않고 ID/N/V 블록으로 분해해 평가합니다.

- `ID-only`, `N-only`, `V-only`, `ID+N`, `ID+V`, `full` Recall/NDCG @10·20·50
- ID 대비 N/V 점수 표준편차(실효강도)
- qN/qV 고유값·동점·상관
- 미래 구매자 조건부 V 상관

이 노트북은 optimizer나 epoch 학습을 호출하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'c74f279a6a0b4630f7d3f7cb93b0849eee9f12cd'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
from IPython.display import display
from lightgcn_clv_joint_nv import configure_joint_nv_run, preflight_summary, run_checkpoint_diagnostics

cfg = configure_joint_nv_run('hm', short_hm=True)
print(json.dumps({**preflight_summary(cfg), 'mode': 'checkpoint_diagnostics_only', 'training': False}, ensure_ascii=False, indent=2))


In [ ]:
diagnostic_df = run_checkpoint_diagnostics(cfg)
columns = [
    'view', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'arp@10',
    'coverage@10', 'n_distinct@10', 'eff_catalog@10',
]
display(diagnostic_df[columns])


In [ ]:
print('γ 및 블록별 실효강도:')
print(json.dumps(diagnostic_df.attrs['block_score_summary'], ensure_ascii=False, indent=2))
print('\nqN/qV 분포:')
print(json.dumps(diagnostic_df.attrs['axis_distribution'], ensure_ascii=False, indent=2))
print('\n기존 결과의 외부 M1:')
print(json.dumps(diagnostic_df.attrs['external_m1'], ensure_ascii=False, indent=2))
print('\n저장 파일:')
print(json.dumps(diagnostic_df.attrs['result_paths'], ensure_ascii=False, indent=2))
